In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_insurance_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "InsuranceId string, SourceSystem string, "
        # "IngestionTimestamp timestamp, BatchId string, InsuranceKey string, RealEstateKey int, InsuranceTypeCode string, "
        # "InsuranceType string, AgreementNumber string, DocumentNumber string, CompanyKey int, GroupInsuranceFlag boolean, "
        # "BrokerKey int,InsurancePool string, MasterCode int, NewCode int, IsActiveFlag boolean, AccrualCode string, "
        # "Addendum boolean, Comments string, InsuranceEnteredDate timestamp, InsuranceEnteredBy int, InsuranceIssuedDate timestamp, "
        # "InsuranceStartDate timestamp, InsuranceEndDate timestamp, InsuranceReceivedDate timestamp, InsurancePaidDate timestamp, "
        # "Description string, TotalAmount decimal(19,4), Action int, BuildingCapital decimal(19,4), InsuranceRealEstateComments string, "
        # "AgreementRightAmount decimal(19,4), DangersCode string, InsuranceRealEstateDateEntered timestamp, InsuranceRealEstateLastUpdatedDate timestamp, "
        # "InsuranceRealEstateDatePaidFrom timestamp, InsuranceRealEstateDatePaidTo timestamp, InsuranceRealEstateEnteredBy int, "
        # "EquipmentCapital decimal(19,4), ForecastInsuranceAmount decimal(19,4), GrossInsuranceCapital decimal(19,4), "
        # "GrossInsuranceAmount decimal(19,4), GrossRate double, ImprovementCapital decimal(19,4), InsuranceTax10Amt decimal(19,4), "
        # "InsuranceTax20Amt decimal(19,4), NetInsuranceCapital decimal(19,4), NetInsuranceAmount decimal(19,4), NetRate double, "
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Insurance"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== prinsurance ==================

        prinsurance_extra_filters = None

        if prinsurance_extra_filters is not None:
            prinsurance_extra_filters.append(date_filter)
            prinsurance_final_filters = " AND ".join(prinsurance_extra_filters)
        else:
            prinsurance_final_filters = date_filter

        prinsurance_selected_fields = [
            "id",
            "type_code",
            "contract_no",
            "document_no",
            "company_id",
            "group_insurance",
            "broker_id",
            "insurance_pool",
            "master_id",
            "new_id",
            "active",
            "accrual_code",
            "addendum",
            "comments",
            "date_entered",
            "entered_by",
            "date_issued",
            "date_start",
            "date_end",
            "date_received",
            "date_paid",
            "descr",
            "total_amount",
            "businessdate",
        ]

        prinsurance = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prinsurance",  # check name
            prinsurance_final_filters,
            prinsurance_selected_fields,
        )

        # Rename "id" column to "prinsurance_id"
        prinsurance = prinsurance.withColumnRenamed("id", "prinsurance_id")

        # ================== prinsurance_det ==================

        prinsurance_det_extra_filters = None

        if prinsurance_det_extra_filters is not None:
            prinsurance_det_extra_filters.append(date_filter)
            prinsurance_det_final_filters = " AND ".join(prinsurance_det_extra_filters)
        else:
            prinsurance_det_final_filters = date_filter

        prinsurance_det_selected_fields = [
            "id",
            "property_id",
            "action",
            "building_capital",
            "comments",
            "contract_right_amt",
            "dangers_code",
            "date_entered",
            "date_modified",
            "date_paid_from",
            "date_paid_to",
            "entered_by",
            "equipment_capital",
            "forecast_insur_amt",
            "gross_ins_capital",
            "gross_insur_amt",
            "gross_rate",
            "improvement_capital",
            "insurance_tax_10_amt",
            "insurance_tax_20_amt",
            "net_ins_capital",
            "net_insur_amt",
            "net_rate",
            "insurance_id",
        ]

        prinsurance_det = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prinsurance_det",  # check name
            prinsurance_det_final_filters,
            prinsurance_det_selected_fields,
        )

        # Rename "id" column to "prinsurance_det_id"
        prinsurance_det = prinsurance_det.withColumnRenamed("id", "prinsurance_det_id")

        # ================== Join ==================
        # Join Condition
        prinsurance_prinsurance_det_join_condition = (
            prinsurance["prinsurance_id"] == prinsurance_det["insurance_id"]
        )

        # Join
        result_df = prinsurance.join(
            prinsurance_det, prinsurance_prinsurance_det_join_condition, "left"
        )

        # HashKeys
        result_df_primary_key_fields = [
            result_df.prinsurance_id,
            result_df.prinsurance_det_id,
            lit("thesis"),
        ]

        # Add columns ("InsuranceId", "InsuranceKey") to final df
        result_df = result_df.withColumn(
            "InsuranceId",
            sha2(concat_ws(";", *result_df_primary_key_fields), 256),
        ).withColumn(
            "InsuranceKey", concat_ws(";", "prinsurance_id", "prinsurance_det_id")
        )

        # ================== Selection of Final Columns ==================

        result_df = result_df.select(
            result_df.InsuranceId.cast("string"),
            lit("thesis").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            result_df.InsuranceKey.cast("string"),
            result_df.property_id.alias("RealEstateKey"),
            result_df.type_code.alias("InsuranceTypeCode"),
            result_df.type_code.alias("InsuranceType"),
            result_df.contract_no.alias("AgreementNumber"),
            result_df.document_no.alias("DocumentNumber"),
            result_df.company_id.alias("CompanyKey"),
            result_df.group_insurance.alias("GroupInsuranceFlag"),
            result_df.broker_id.alias("BrokerKey"),
            result_df.insurance_pool.alias("InsurancePool"),
            result_df.master_id.alias("MasterCode"),
            result_df.new_id.alias("NewCode"),
            result_df.active.alias("IsActiveFlag"),
            result_df.accrual_code.alias("AccrualCode"),
            result_df.addendum.alias("Addendum"),
            prinsurance.comments.alias("Comments"),
            prinsurance.date_entered.alias("InsuranceEnteredDate"),
            prinsurance.entered_by.alias("InsuranceEnteredBy"),
            result_df.date_issued.alias("InsuranceIssuedDate"),
            result_df.date_start.alias("InsuranceStartDate"),
            result_df.date_end.alias("InsuranceEndDate"),
            result_df.date_received.alias("InsuranceReceivedDate"),
            result_df.date_paid.alias("InsurancePaidDate"),
            result_df.descr.alias("Description"),
            result_df.total_amount.alias("TotalAmount"),
            result_df.action.alias("Action"),
            result_df.building_capital.alias("BuildingCapital"),
            prinsurance_det.comments.alias("InsuranceRealEstateComments"),
            result_df.contract_right_amt.alias("AgreementRightAmount"),
            result_df.dangers_code.alias("DangersCode"),
            prinsurance_det.date_entered.alias("InsuranceRealEstateDateEntered"),
            result_df.date_modified.alias("InsuranceRealEstateLastUpdatedDate"),
            result_df.date_paid_from.alias("InsuranceRealEstateDatePaidFrom"),
            result_df.date_paid_to.alias("InsuranceRealEstateDatePaidTo"),
            prinsurance_det.entered_by.alias("InsuranceRealEstateEnteredBy"),
            result_df.equipment_capital.alias("EquipmentCapital"),
            result_df.forecast_insur_amt.alias("ForecastInsuranceAmount"),
            result_df.gross_ins_capital.alias("GrossInsuranceCapital"),
            result_df.gross_insur_amt.alias("GrossInsuranceAmount"),
            result_df.gross_rate.alias("GrossRate"),
            result_df.improvement_capital.alias("ImprovementCapital"),
            result_df.insurance_tax_10_amt.alias("InsuranceTax10Amt"),
            result_df.insurance_tax_20_amt.alias("InsuranceTax20Amt"),
            result_df.net_ins_capital.alias("NetInsuranceCapital"),
            result_df.net_insur_amt.alias("NetInsuranceAmount"),
            result_df.net_rate.alias("NetRate"),
            result_df.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_insurance_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)